In [1]:
#Date 24th 24 Jun — Calculate total unique invoices per customer Goal
import pandas as pd
df = pd.read_csv("C:/Users/yash6/Internship 1/cleaned_retail_data_updated.csv", encoding='latin-1')
print(df.head())
print(df.shape)


   InvoiceNo StockCode                          Description  Quantity  \
0     536365    85123A   WHITE HANGING HEART T-LIGHT HOLDER         6   
1     536365     71053                  WHITE METAL LANTERN         6   
2     536365    84406B       CREAM CUPID HEARTS COAT HANGER         8   
3     536365    84029G  KNITTED UNION FLAG HOT WATER BOTTLE         6   
4     536365    84029E       RED WOOLLY HOTTIE WHITE HEART.         6   

        InvoiceDate  UnitPrice  CustomerID         Country TransactionMonth  \
0  01-12-2010 08:26       2.55       17850  United Kingdom       01-12-2010   
1  01-12-2010 08:26       3.39       17850  United Kingdom       01-12-2010   
2  01-12-2010 08:26       2.75       17850  United Kingdom       01-12-2010   
3  01-12-2010 08:26       3.39       17850  United Kingdom       01-12-2010   
4  01-12-2010 08:26       3.39       17850  United Kingdom       01-12-2010   

  CohortMonth  CohortIndex  
0  01-12-2010          0.0  
1  01-12-2010          0.0  

In [2]:
df = df.dropna(subset=["CustomerID"])

In [3]:
df["CustomerID"] = df["CustomerID"].astype(int)

In [4]:
customer_frequency = (
    df.groupby("CustomerID")["InvoiceNo"]
    .nunique()
    .reset_index()
)


In [5]:
customer_frequency.columns = ["CustomerID", "PurchaseFrequency"]

In [6]:
import os
# Create outputs folder
output_folder = "outputs"
os.makedirs(output_folder, exist_ok=True)

In [7]:
file_path = os.path.join(
    output_folder,
    "customer_purchase_frequency.csv"
)
customer_frequency.to_csv(file_path, index=False)


In [8]:
print("Successfully saved!")
print("File location:", os.path.abspath(file_path))
print("Total customers:", len(customer_frequency))

display(customer_frequency.head())

Successfully saved!
File location: C:\Users\yash6\Internship 1\outputs\customer_purchase_frequency.csv
Total customers: 4338


,CustomerID,PurchaseFrequency
0,12346,1
1,12347,7
2,12348,4
3,12349,1
4,12350,1


In [9]:
#25 Jun — Calculate average purchase frequency per cohort segment

In [11]:
customer_cohort = (
    df.groupby("CustomerID")["CohortMonth"]
    .min()
    .reset_index()
)

In [12]:
frequency_with_cohort = customer_frequency.merge(
    customer_cohort,
    on="CustomerID",
    how="left"
)

In [13]:
cohort_frequency = (
    frequency_with_cohort.groupby("CohortMonth")["PurchaseFrequency"]
    .agg(
        AveragePurchaseFrequency="mean",
        TotalCustomers="count",
        MedianPurchaseFrequency="median"
    )
    .reset_index()
)


In [14]:
cohort_frequency["AveragePurchaseFrequency"] = (
    cohort_frequency["AveragePurchaseFrequency"].round(2)
)

print(cohort_frequency.head())

  CohortMonth  AveragePurchaseFrequency  TotalCustomers  \
0  01-01-2011                      6.83             178   
1  01-02-2011                      5.78             190   
2  01-03-2011                      4.67             234   
3  01-04-2011                      5.09             232   
4  01-05-2011                      4.74             253   

   MedianPurchaseFrequency  
0                      5.0  
1                      4.0  
2                      3.5  
3                      4.0  
4                      4.0  


In [16]:
import os
output_folder = r"C:\Users\yash6\Internship 1\outputs"

# Create it automatically if missing
os.makedirs(output_folder, exist_ok=True)

# Correct file location
output_file = os.path.join(
    output_folder,
    "cohort_purchase_frequency.csv"
)


In [17]:
cohort_frequency.to_csv(output_file, index=False)

print("Saved successfully!")
print("File location:", output_file)

Saved successfully!
File location: C:\Users\yash6\Internship 1\outputs\cohort_purchase_frequency.csv


In [ ]:
#26th Jun — Flag high-frequency vs low-frequency customer segments

In [18]:
def frequency_segment(value):
    if value == 1:
        return "Low Frequency"
    elif 2 <= value <= 4:
        return "Medium Frequency"
    else:
        return "High Frequency"

In [19]:
frequency_with_cohort["FrequencySegment"] = (
    frequency_with_cohort["PurchaseFrequency"]
    .apply(frequency_segment)
)

In [20]:
print(frequency_with_cohort.head())

   CustomerID  PurchaseFrequency CohortMonth  FrequencySegment
0       12346                  1         NaN     Low Frequency
1       12347                  7  01-12-2010    High Frequency
2       12348                  4  01-04-2011  Medium Frequency
3       12349                  1         NaN     Low Frequency
4       12350                  1  01-02-2011     Low Frequency


In [21]:
frequency_segment_summary = (
    frequency_with_cohort.groupby("FrequencySegment")
    .agg(
        TotalCustomers=("CustomerID", "count"),
        AveragePurchaseFrequency=("PurchaseFrequency", "mean"),
        MinPurchaseFrequency=("PurchaseFrequency", "min"),
        MaxPurchaseFrequency=("PurchaseFrequency", "max")
    )
    .reset_index()
)

frequency_segment_summary["AveragePurchaseFrequency"] = (
    frequency_segment_summary["AveragePurchaseFrequency"].round(2)
)

print(frequency_segment_summary)

   FrequencySegment  TotalCustomers  AveragePurchaseFrequency  \
0    High Frequency            1114                     11.04   
1     Low Frequency            1493                      1.00   
2  Medium Frequency            1731                      2.74   

   MinPurchaseFrequency  MaxPurchaseFrequency  
0                     5                   209  
1                     1                     1  
2                     2                     4  


In [22]:
import os
output_folder = r"C:\Users\yash6\Internship 1\outputs"
# Create outputs folder if it does not exist
os.makedirs(output_folder, exist_ok=True)
customer_segments_file = os.path.join(
    output_folder,
    "customer_frequency_segments.csv"
)
segment_summary_file = os.path.join(
    output_folder,
    "frequency_segment_summary.csv"
)


In [23]:
frequency_with_cohort.to_csv(customer_segments_file, index=False)
frequency_segment_summary.to_csv(segment_summary_file, index=False)
print("Day 3 files saved successfully!")
print("Customer segments file:", customer_segments_file)
print("Segment summary file:", segment_summary_file)

Day 3 files saved successfully!
Customer segments file: C:\Users\yash6\Internship 1\outputs\customer_frequency_segments.csv
Segment summary file: C:\Users\yash6\Internship 1\outputs\frequency_segment_summary.csv


In [ ]:
#27th Jun — Validate and cross-check purchase frequency results

In [24]:
original_unique_customers = df["CustomerID"].nunique()

In [25]:
frequency_table_customers = customer_frequency["CustomerID"].nunique()


In [26]:
segmented_table_customers = frequency_with_cohort["CustomerID"].nunique()

In [27]:
validation_results = pd.DataFrame({
    "Check": [
        "Unique customers in original dataset",
        "Unique customers in purchase frequency table",
        "Unique customers in frequency segment table",
        "Customer count match"
    ],
    "Result": [
        original_unique_customers,
        frequency_table_customers,
        segmented_table_customers,
        original_unique_customers == frequency_table_customers == segmented_table_customers
    ]
})

print(validation_results)

                                          Check Result
0          Unique customers in original dataset   4338
1  Unique customers in purchase frequency table   4338
2   Unique customers in frequency segment table   4338
3                          Customer count match   True


In [28]:
duplicate_customers = customer_frequency["CustomerID"].duplicated().sum()

print("Duplicate customers in frequency table:", duplicate_customers)

Duplicate customers in frequency table: 0


In [29]:
missing_values = frequency_with_cohort.isnull().sum()

print("\nMissing values:")
print(missing_values)


Missing values:
CustomerID              0
PurchaseFrequency       0
CohortMonth          1341
FrequencySegment        0
dtype: int64


In [30]:
segment_total = frequency_segment_summary["TotalCustomers"].sum()

print("\nSegment total:", segment_total)
print("Original customer total:", original_unique_customers)


Segment total: 4338
Original customer total: 4338


In [31]:
import os
# Your output folder
output_folder = r"C:\Users\yash6\Internship 1\outputs"
# Create folder if it does not exist
os.makedirs(output_folder, exist_ok=True)
# Full path for validation file
validation_file = os.path.join(
    output_folder,
    "validation_results.csv"
)

In [32]:
validation_results.to_csv(validation_file, index=False)
print("Day 4 validation file saved successfully!")
print("Saved at:", validation_file)

Day 4 validation file saved successfully!
Saved at: C:\Users\yash6\Internship 1\outputs\validation_results.csv


In [ ]:
#29 Jun — Verify outputs ready for Member D CLTV inputs and push

In [5]:
import pandas as pd
frequency_with_cohort = pd.read_csv(
    r"C:\Users\yash6\Internship 1\outputs\customer_frequency_segments.csv"
)
print(frequency_with_cohort.head())

   CustomerID  PurchaseFrequency CohortMonth  FrequencySegment
0       12346                  1         NaN     Low Frequency
1       12347                  7  01-12-2010    High Frequency
2       12348                  4  01-04-2011  Medium Frequency
3       12349                  1         NaN     Low Frequency
4       12350                  1  01-02-2011     Low Frequency


In [6]:
member_d_cltv_input = frequency_with_cohort[
    [
        "CustomerID",
        "CohortMonth",
        "PurchaseFrequency",
        "FrequencySegment"
    ]
].copy()

print(member_d_cltv_input.head())

   CustomerID CohortMonth  PurchaseFrequency  FrequencySegment
0       12346         NaN                  1     Low Frequency
1       12347  01-12-2010                  7    High Frequency
2       12348  01-04-2011                  4  Medium Frequency
3       12349         NaN                  1     Low Frequency
4       12350  01-02-2011                  1     Low Frequency


In [7]:
member_d_cltv_input = member_d_cltv_input.sort_values(
    by="CustomerID"
)

In [8]:
print("Rows:", member_d_cltv_input.shape[0])
print("Unique Customers:", member_d_cltv_input["CustomerID"].nunique())
print("\nMissing Values:")
print(member_d_cltv_input.isnull().sum())

Rows: 4338
Unique Customers: 4338

Missing Values:
CustomerID              0
CohortMonth          1341
PurchaseFrequency       0
FrequencySegment        0
dtype: int64


In [9]:
import os
output_folder = r"C:\Users\yash6\Internship 1\outputs"
os.makedirs(output_folder, exist_ok=True)
member_d_cltv_input.to_csv(
    os.path.join(output_folder, "member_d_cltv_input.csv"),
    index=False
)
print("Day 5 completed successfully!")

Day 5 completed successfully!
